# Junção: crédito rural (BCB) e produção agrícola (IBGE)

Este notebook **não define funções de coleta**. Ele executa `API_BC.ipynb` e `API_IBGE.ipynb` no mesmo kernel com `%run`, usa o que eles produzem e junta tudo em uma única tabela.

**Chave de junção:** `codigo_municipio` (código IBGE de 7 dígitos, como texto) + `cultura` (`soja`, `milho`, `cana_de_acucar`) + `ano`.

**Pré-requisitos:** os dois notebooks e a pasta `certs` devem estar na mesma pasta deste arquivo. `API_BC.ipynb` e `API_IBGE.ipynb` precisam rodar até o fim sem erro, pois o `%run` interrompe a execução na primeira falha. Execute as células em ordem, de um kernel reiniciado.

**Resultado:** `tabela`, com uma linha por município–cultura–ano, salva em `dados/tabela_credito_producao.csv`.

### 1. Execução do notebook do Banco Central

`%run` executa todas as células de código de `API_BC.ipynb` neste kernel. Com isso ficam disponíveis `coletar_periodo`, `agregar_anual`, `RECURSO_MUNICIPAL` e `produtos`, entre outros nomes definidos lá.

In [1]:
%run API_BC.ipynb

,recurso,campo,tipo
0,CusteioMunicipioProduto,Municipio,Edm.String
1,CusteioMunicipioProduto,nomeProduto,Edm.String
2,CusteioMunicipioProduto,MesEmissao,Edm.String
3,CusteioMunicipioProduto,AnoEmissao,Edm.String
4,CusteioMunicipioProduto,cdPrograma,Edm.String
5,CusteioMunicipioProduto,cdSubPrograma,Edm.String
6,CusteioMunicipioProduto,cdFonteRecurso,Edm.String
7,CusteioMunicipioProduto,cdTipoSeguro,Edm.String
8,CusteioMunicipioProduto,cdEstado,Edm.String
9,CusteioMunicipioProduto,VlCusteio,Edm.Decimal


Quantidade de contratos no recurso municipal: False


,nomeProduto,cdProduto,Atividade,cultura
0,"""SOJA""",6720,1,soja
1,"""MILHO""",5080,1,milho
2,"""CANA-DE-AÇUCAR""",1840,1,cana_de_acucar


### 2. Execução do notebook do IBGE

Executa `API_IBGE.ipynb` da mesma forma. Ficam disponíveis `df_area_municipal`, `df_prod_municipal` e `df_rendimento_municipal`, de 2016 a 2025.

**Atenção:** os dois notebooks usam alguns nomes em comum (`dados`, `url`, `parametros`). Por isso o BCB roda antes, e a coleta do crédito (passo 4) usa só nomes que o IBGE não sobrescreve.

In [2]:
%run API_IBGE.ipynb

Tabela: Área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias
109 Área plantada Hectares
1000109 Área plantada - percentual do total geral %
216 Área colhida Hectares
1000216 Área colhida - percentual do total geral %
214 Quantidade produzida Toneladas
112 Rendimento médio da produção Quilogramas por Hectare
215 Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992], Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
1000215 Valor da produção - percentual do total geral %
Classificação: 81 Produto das lavouras temporárias
0 Total
2688 Abacaxi*
83374 Abóbora
40471 Alfafa fenada
83375 Alface
2689 Algodão herbáceo (em caroço)
2690 Alho
2691 Amendoim (em casca)
2692 Arroz (em casca)
2693 Aveia (em grão)
2694 Batata-doce
2695 Batata-inglesa
2696 Cana-de-açúcar
40470 Cana para forragem
83384 Canola
2697 Cebola
83381 Cenoura
2698 Centeio (em grão)
2699 Cevada (em grão)
83379 

### 3. Bibliotecas da junção

In [3]:
import pandas as pd
from pathlib import Path

### 4. Coleta do crédito (usa as funções de `API_BC.ipynb`)

`API_BC.ipynb` apenas define as funções de coleta; esta célula as chama para o período 2016–2025 (360 consultas: 10 anos × 12 meses × 3 produtos). A primeira execução é demorada, e as seguintes usam o cache em `dados/bcb/cache`.

Resultado: `credito_anual`, com o crédito nominal (em reais correntes, sem deflacionar) por município, cultura e ano.

In [4]:
ANOS = range(2016, 2026)

dados_bcb, auditoria_bcb = coletar_periodo(RECURSO_MUNICIPAL, ANOS, produtos)
credito_anual = agregar_anual(dados_bcb, RECURSO_MUNICIPAL)

display(credito_anual.head())
print(credito_anual.shape)

CusteioMunicipioProduto: 2016 consultado.


RuntimeError: Falha ao consultar BCB: https://olinda.bcb.gov.br/olinda/servico/SICOR/versao/v2/odata/CusteioMunicipioProduto?%24format=json&%24top=100000&%24filter=AnoEmissao%20eq%20%272017%27%20and%20MesEmissao%20eq%20%2707%27%20and%20Atividade%20eq%20%271%27%20and%20nomeProduto%20eq%20%27%22CANA-DE-A%C3%87UCAR%22%27
HTTPSConnectionPool(host='olinda.bcb.gov.br', port=443): Max retries exceeded with url: /olinda/servico/SICOR/versao/v2/odata/CusteioMunicipioProduto?%24format=json&%24top=100000&%24filter=AnoEmissao%20eq%20%272017%27%20and%20MesEmissao%20eq%20%2707%27%20and%20Atividade%20eq%20%271%27%20and%20nomeProduto%20eq%20%27%22CANA-DE-A%C3%87UCAR%22%27 (Caused by ResponseError('too many 500 error responses'))

### 5. Medidas do IBGE em uma única tabela

Os três DataFrames do IBGE vêm em formato longo (uma linha por medida). Aqui eles são colocados lado a lado, em colunas, usando as mesmas chaves da junção. A cultura é identificada pelo código da classificação 81 (`2713` soja, `2711` milho, `2696` cana-de-açúcar), e não pelo nome, que no IBGE é "Soja (em grão)" e no BCB é "SOJA".

As asserções interrompem a execução se houver cultura sem mapeamento, código de município que não tenha 7 dígitos ou chaves duplicadas.

**Valores ausentes:** `API_IBGE.ipynb` descarta os registros que não são numéricos (`..`, `...`, `X`). Esses casos aparecem aqui como ausentes (`NaN`) depois da junção.

In [ ]:
CHAVES = ["codigo_municipio", "cultura", "ano"]
cultura_por_id = {"2713": "soja", "2711": "milho", "2696": "cana_de_acucar"}

medidas = {
    "area_plantada_ha": df_area_municipal,
    "producao_t": df_prod_municipal,
    "rendimento_kg_ha": df_rendimento_municipal,
}

ibge = None
nomes = []
for nome, df in medidas.items():
    df = df.copy()
    df["cultura"] = df["codigo_cultura"].astype(str).map(cultura_por_id)
    df["codigo_municipio"] = df["codigo_municipio"].astype("string").str.strip()
    df["ano"] = df["ano"].astype("int64")
    assert df["cultura"].notna().all(), f"Cultura sem mapeamento em {nome}"
    assert df["codigo_municipio"].str.fullmatch(r"\d{7}").all(), f"Código de município inválido em {nome}"
    assert not df.duplicated(CHAVES).any(), f"Chaves duplicadas em {nome}"
    nomes.append(df[["codigo_municipio", "municipio"]])
    parte = df[CHAVES + ["valor"]].rename(columns={"valor": nome})
    ibge = parte if ibge is None else ibge.merge(parte, on=CHAVES, how="outer")

nomes = pd.concat(nomes).drop_duplicates("codigo_municipio")

display(ibge.head())
print(ibge.shape)

### 6. Junção com o crédito

Une as medidas do IBGE ao crédito e acrescenta a coluna `cobertura`:

| Valor | Significado |
|---|---|
| `ambos` | o município–cultura–ano existe no BCB e no IBGE |
| `so_credito` | há crédito, mas o IBGE não tem registro |
| `so_ibge` | há registro no IBGE, mas não há crédito no BCB |

A junção é `outer` para que nada desapareça sem aviso. Ausência de linha não equivale a zero: município sem registro no IBGE não é município com produção zero, e município sem crédito no BCB não é crédito zero. Municípios que só têm crédito ficam sem o nome (`municipio`), pois o nome vem do IBGE.

Também são calculados `credito_por_ha` e `credito_por_t`, que ficam ausentes quando a área ou a produção é zero ou ausente.

In [ ]:
credito = credito_anual[CHAVES + ["credito_nominal_brl"]].copy()
credito["codigo_municipio"] = credito["codigo_municipio"].astype("string").str.strip()
credito["ano"] = credito["ano"].astype("int64")
credito["credito_nominal_brl"] = credito["credito_nominal_brl"].astype("float64")
assert not credito.duplicated(CHAVES).any(), "Chaves duplicadas no crédito"

tabela = ibge.merge(credito, on=CHAVES, how="outer", indicator="cobertura")
tabela["cobertura"] = tabela["cobertura"].astype(str).map(
    {"both": "ambos", "left_only": "so_ibge", "right_only": "so_credito"})

tabela = tabela.merge(nomes, on="codigo_municipio", how="left")
tabela["codigo_uf_ibge"] = tabela["codigo_municipio"].str[:2]

area = tabela["area_plantada_ha"]
producao = tabela["producao_t"]
tabela["credito_por_ha"] = tabela["credito_nominal_brl"] / area.where(area > 0)
tabela["credito_por_t"] = tabela["credito_nominal_brl"] / producao.where(producao > 0)

primeiras = ["codigo_municipio", "municipio", "codigo_uf_ibge", "cultura", "ano"]
resto = [c for c in tabela.columns if c not in primeiras]
tabela = tabela[primeiras + resto].sort_values(CHAVES).reset_index(drop=True)

display(tabela.head())
print(tabela.shape)

### 7. Verificação

Confere quantas linhas casaram, a cobertura por cultura e ano e os valores ausentes. Uma prévia com `head()` não valida a cobertura completa; é nestes números que se decide, por exemplo, se as análises usarão apenas `cobertura == "ambos"`.

**Atenção metodológica:** o BCB agrupa o crédito pelo ano de emissão do contrato (`AnoEmissao`), e a PAM registra o ano da colheita. Um custeio contratado em 2024 para a safra 24/25, colhida em 2025, aparece em anos diferentes nas duas bases.

In [ ]:
print(tabela["cobertura"].value_counts())
print()
print("Municípios por cultura e ano (cobertura 'ambos'):")
display(tabela[tabela["cobertura"].eq("ambos")].pivot_table(
    index="cultura", columns="ano", values="codigo_municipio", aggfunc="count"))
print()
print("Valores ausentes por coluna:")
display(tabela.isna().sum())

### 8. Exportação

Salva a tabela final em CSV. Ao relê-la, use `dtype={"codigo_municipio": str}` para preservar o código como texto.

In [ ]:
Path("dados").mkdir(exist_ok=True)
tabela.to_csv("dados/tabela_credito_producao.csv", index=False, encoding="utf-8")
print("Salvo em dados/tabela_credito_producao.csv")
tabela.to_csv.head()